# Old

In [ ]:
FOR Neo4j Version 4.XX

In [15]:
# Install Neo4j driver if needed
# !pip install neo4j

from neo4j import GraphDatabase
import json

# ----------- CONFIGURATION -----------
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "password"  # Change to your real password!

# kg_id = "bioKG"
# kg_name = "bioKG"
# kg_id = "primeKG"
# kg_name = "Precision Medicine Knowledge Graph"
# kg_id = "PharMeBINet"
# kg_name = "heterogeneous pharmacological medical biochemical network"
# kg_id = "CKG (incomplete)"
# kg_name = "Clinical Knowledge Graph (incomplete)"
# kg_id = "OREGANO_V2"
# kg_name = "OREGANO"
kg_id = "monarchKgG_V4.4.42"
kg_name = "monarchKg"
out_path = f"{kg_id}_stats.json"

# ----------- CONNECT -----------
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# ----------- STATISTICS FUNCTIONS -----------

def get_node_counts_multilabel(driver):
    """
    Counts nodes by their *complete label combination* (as a tuple/list).
    Example: [{'type': ['Protein'], 'count': 500}, {'type': ['Protein', 'Enzyme'], 'count': 50}]
    """
    with driver.session() as session:
        query = """
        MATCH (n)
        WITH labels(n) AS labels
        RETURN labels, count(*) AS count
        ORDER BY count DESC
        """
        results = []
        for rec in session.run(query):
            results.append({"type": rec["labels"], "count": rec["count"]})
    return results

def get_node_counts_aggregated(driver):
    """
    Counts nodes using the same aggregation as the Sankey aggregation.
    Each node is mapped to a "primary" label (single or synthetic).
    Returns: [{'type': [aggregated_label], 'count': ...}, ...]
    """
    # Collect aggregation rules, same as in get_sankey_links_aggregated
    with driver.session() as session:
        label_to_ids = {}
        node_labels = []
        result = session.run("MATCH (n) RETURN labels(n) AS labels")
        for rec in result:
            labels = rec["labels"]
            node_labels.append(labels)
            for l in labels:
                label_to_ids.setdefault(l, set()).add(tuple(labels))
        single_primaries = set()
        for labels in node_labels:
            if len(labels) == 1:
                single_primaries.add(labels[0])
        synthetic_primaries = set()
        for l, combos in label_to_ids.items():
            if all(len(combo) > 1 for combo in combos):
                synthetic_primaries.add(l)

        # Now map each node to its aggregated label, then count
        agg_counts = {}
        result = session.run("MATCH (n) RETURN labels(n) AS labels")
        for rec in result:
            labels = rec["labels"]
            # Aggregation logic (as in sankey)
            agg = None
            for l in labels:
                if l in single_primaries:
                    agg = l
                    break
            if agg is None:
                for l in labels:
                    if l in synthetic_primaries:
                        agg = l
                        break
            if agg is None and labels:
                agg = labels[0]
            # Count
            key = (agg,) if agg else tuple(labels)
            agg_counts[key] = agg_counts.get(key, 0) + 1
        # Convert to desired format
        return [{"type": list(key), "count": count} for key, count in agg_counts.items()]
        
def get_out_degree_distribution(driver):
    query = """
    MATCH (n)
    WITH size([(n)-->() | 1]) AS out_degree
    RETURN out_degree AS degree, count(*) AS count
    ORDER BY degree
    """
    with driver.session() as session:
        records = session.run(query)
        return [{"degree": r["degree"], "count": r["count"]} for r in records]


def get_in_degree_distribution(driver):
    query = """
    MATCH (n)
    WITH size([()-->(n) | 1]) AS in_degree
    RETURN in_degree AS degree, count(*) AS count
    ORDER BY degree
    """
    with driver.session() as session:
        records = session.run(query)
        return [{"degree": r["degree"], "count": r["count"]} for r in records]


def get_relationship_counts(driver):
    with driver.session() as session:
        rels = session.run("CALL db.relationshipTypes() YIELD relationshipType AS relType RETURN relType")
        results = []
        for rec in rels:
            rel_type = rec["relType"]
            count = session.run(f"MATCH ()-[:`{rel_type}`]->() RETURN count(*) AS count").single()["count"]
            results.append({"type": rel_type, "count": count})
    return results

def get_out_degree_distribution_by_label(driver):
    results = []
    with driver.session() as session:
        query = """
        MATCH (n)
        WITH labels(n) AS lbls, size([(n)-->() | 1]) AS out_degree
        RETURN lbls AS labels, out_degree AS degree, count(*) AS count
        ORDER BY labels, degree
        """
        combo_dict = {}
        for r in session.run(query):
            key = tuple(r["labels"])
            if key not in combo_dict:
                combo_dict[key] = []
            combo_dict[key].append({"degree": r["degree"], "count": r["count"]})
        for labels, degs in combo_dict.items():
            results.append({"labels": list(labels), "degrees": degs})
    return results


def get_in_degree_distribution_by_label(driver):
    """
    For each label, computes in-degree distribution (direct only).
    Returns: list of {label: str, degrees: [{degree: int, count: int}, ...]}
    """
    results = []
    with driver.session() as session:
        labels = [rec["label"] for rec in session.run("CALL db.labels() YIELD label RETURN label")]
        for label in labels:
            q = f"""
            MATCH (n:`{label}`)
            WITH size([()-->(n) | 1]) AS in_degree
            RETURN in_degree AS degree, count(*) AS count
            ORDER BY degree
            """
            degs = [{"degree": r["degree"], "count": r["count"]} for r in session.run(q)]
            results.append({"label": label, "degrees": degs})
    return results
    
def get_out_degree_distribution_by_label(driver):
    results = []
    with driver.session() as session:
        labels = [rec["label"] for rec in session.run("CALL db.labels() YIELD label RETURN label")]
        for label in labels:
            q = f"""
            MATCH (n:`{label}`)
            WITH size([(n)-->() | 1]) AS out_degree
            RETURN out_degree AS degree, count(*) AS count
            ORDER BY degree
            """
            degs = [{"degree": r["degree"], "count": r["count"]} for r in session.run(q)]
            results.append({"label": label, "degrees": degs})
    return results

def get_out_degree_distribution_by_label_combo(driver):
    results = []
    with driver.session() as session:
        query = """
        MATCH (n)
        WITH labels(n) AS lbls, size([(n)-->() | 1]) AS out_degree
        RETURN lbls AS labels, out_degree AS degree, count(*) AS count
        ORDER BY labels, degree
        """
        combo_dict = {}
        for r in session.run(query):
            key = tuple(r["labels"])
            if key not in combo_dict:
                combo_dict[key] = []
            combo_dict[key].append({"degree": r["degree"], "count": r["count"]})
        for labels, degs in combo_dict.items():
            results.append({"labels": list(labels), "degrees": degs})
    return results

def get_in_degree_distribution_by_label_combo(driver):
    """
    For each multi-label combo, computes in-degree distribution.
    Returns: list of {"labels": [...], "degrees": [{"degree": int, "count": int}, ...]}
    """
    results = []
    with driver.session() as session:
        query = """
        MATCH (n)
        WITH labels(n) AS lbls, size([()-->(n) | 1]) AS in_degree
        RETURN lbls AS labels, in_degree AS degree, count(*) AS count
        ORDER BY labels, degree
        """
        combo_dict = {}
        for r in session.run(query):
            key = tuple(r["labels"])
            if key not in combo_dict:
                combo_dict[key] = []
            combo_dict[key].append({"degree": r["degree"], "count": r["count"]})
        for labels, degs in combo_dict.items():
            results.append({"labels": list(labels), "degrees": degs})
    return results

def get_sankey_links_unaggregated(driver):
    """
    Returns: List of {
        "source": [label, ...],       # labels of source node
        "target": [label, ...],       # labels of target node
        "relationship": str,          # edge type
        "value": int                  # edge count
    }
    """
    results = []
    with driver.session() as session:
        query = """
        MATCH (a)-[r]->(b)
        WITH labels(a) AS src_labels, type(r) AS rel_type, labels(b) AS tgt_labels, count(*) AS value
        RETURN src_labels, rel_type, tgt_labels, value
        ORDER BY value DESC
        """
        for rec in session.run(query):
            results.append({
                "source": rec["src_labels"],
                "relationship": rec["rel_type"],
                "target": rec["tgt_labels"],
                "value": rec["value"]
            })
    return results


def aggregate_label(labels, single_primaries, synthetic_primaries):
    # Try to map to a real single-label
    for l in labels:
        if l in single_primaries:
            return [l]
    # Else, map to a synthetic primary (first one in list)
    for l in labels:
        if l in synthetic_primaries:
            return [l]
    # Fallback: keep all
    return labels

def get_sankey_links_aggregated(driver):
    """
    Aggregates multi-label nodes using same rules as the frontend's buildAutoIdMap().
    """
    # --- First, collect label stats for aggregation logic ---
    with driver.session() as session:
        # Build label to node mapping
        label_to_ids = {}
        node_labels = []
        result = session.run("MATCH (n) RETURN labels(n) AS labels")
        for rec in result:
            labels = rec["labels"]
            node_labels.append(labels)
            for l in labels:
                label_to_ids.setdefault(l, set()).add(tuple(labels))

        # Find single-primaries (labels that occur alone)
        single_primaries = set()
        for labels in node_labels:
            if len(labels) == 1:
                single_primaries.add(labels[0])
        # Synthetic primaries: labels never occurring alone
        synthetic_primaries = set()
        for l, combos in label_to_ids.items():
            if all(len(combo) > 1 for combo in combos):
                synthetic_primaries.add(l)

        # --- Now fetch all edges with labels and aggregate ---
        results = []
        edge_query = """
        MATCH (a)-[r]->(b)
        RETURN labels(a) AS src_labels, type(r) AS rel_type, labels(b) AS tgt_labels, count(*) AS value
        """
        for rec in session.run(edge_query):
            src_labels = aggregate_label(rec["src_labels"], single_primaries, synthetic_primaries)
            tgt_labels = aggregate_label(rec["tgt_labels"], single_primaries, synthetic_primaries)
            results.append({
                "source": src_labels,
                "relationship": rec["rel_type"],
                "target": tgt_labels,
                "value": rec["value"]
            })
    return results

def get_total_degree_distribution(driver):
    query = """
    MATCH (n)
    WITH size([(n)-->() | 1]) + size([()-->(n) | 1]) AS total_degree
    RETURN total_degree AS degree, count(*) AS count
    ORDER BY degree
    """
    with driver.session() as session:
        records = session.run(query)
        return [{"degree": r["degree"], "count": r["count"]} for r in records]

def get_total_degree_distribution_by_label(driver):
    results = []
    with driver.session() as session:
        labels = [rec["label"] for rec in session.run("CALL db.labels() YIELD label RETURN label")]
        for label in labels:
            q = f"""
            MATCH (n:`{label}`)
            WITH size([(n)-->() | 1]) + size([()-->(n) | 1]) AS total_degree
            RETURN total_degree AS degree, count(*) AS count
            ORDER BY degree
            """
            degs = [{"degree": r["degree"], "count": r["count"]} for r in session.run(q)]
            results.append({"label": label, "degrees": degs})
    return results

def get_total_degree_distribution_by_label_combo(driver):
    """
    For each multi-label combo, computes total degree (in + out) distribution.
    Returns: list of {"labels": [...], "degrees": [{"degree": int, "count": int}, ...]}
    """
    results = []
    with driver.session() as session:
        query = """
        MATCH (n)
        WITH labels(n) AS lbls,
             size([(n)-->() | 1]) + size([()-->(n) | 1]) AS total_degree
        RETURN lbls AS labels, total_degree AS degree, count(*) AS count
        ORDER BY labels, degree
        """
        combo_dict = {}
        for r in session.run(query):
            key = tuple(r["labels"])
            if key not in combo_dict:
                combo_dict[key] = []
            combo_dict[key].append({"degree": r["degree"], "count": r["count"]})
        for labels, degs in combo_dict.items():
            results.append({"labels": list(labels), "degrees": degs})
    return results

def get_zero_degree_node_stats(driver):
    with driver.session() as session:
        # Count of zero degree nodes
        zero_query = "MATCH (n) WHERE NOT (n)--() RETURN count(*) AS zero_degree_nodes"
        zero_count = session.run(zero_query).single()["zero_degree_nodes"]
        # Total nodes
        total_query = "MATCH (n) RETURN count(*) AS total_nodes"
        total_count = session.run(total_query).single()["total_nodes"]
        percent = (zero_count / total_count * 100) if total_count > 0 else 0
        return {
            "zero_degree_nodes": zero_count,
            "total_nodes": total_count,
            "percent": percent
        }

# ----------- JSON WRITER -----------

def save_kg_stats_json(
    kg_id,
    kg_name,
    stats_dict,
    out_path="kg_stats.json"
):
    stats = []
    for stat_id, stat in stats_dict.items():
        data = stat["data"]
        stat_entry = {
            "stat_id": stat_id,
            "data": data
        }
        stats.append(stat_entry)

    out = {
                "kg_id": kg_id,
                "kg_name": kg_name,
                "stats": stats
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)
    print(f"Wrote statistics to {out_path}")

# ----------- RUN EXTRACTIONS AND SAVE -----------

stats_dict = {
    "node_count_per_type": {
        "data": get_node_counts_multilabel(driver),
    },
    "node_count_aggregated": {
    "data": get_node_counts_aggregated(driver),
    },
    "total_degree_distribution": {
        "data": get_total_degree_distribution(driver),
    },
    "out_degree_distribution": {
        "data": get_out_degree_distribution(driver),
    },
    "in_degree_distribution": {
        "data": get_in_degree_distribution(driver),
    },
    "total_degree_by_label": {
       "data": get_total_degree_distribution_by_label(driver),
    },
    "out_degree_by_label": {
        "data": get_out_degree_distribution_by_label(driver),
    },
    "in_degree_by_label": {
        "data": get_in_degree_distribution_by_label(driver),
    },
    "out_degree_by_label_combo": {
        "data": get_out_degree_distribution_by_label_combo(driver),
    },
    "in_degree_by_label_combo": {
        "data": get_in_degree_distribution_by_label_combo(driver),
    },
    "total_degree_by_label_combo": {
        "data": get_total_degree_distribution_by_label_combo(driver),
    },
    "relationship_count_per_type": {
        "data": get_relationship_counts(driver),
    },
    "sankey_links_unaggregated": {
        "data": get_sankey_links_unaggregated(driver)
    },
    "sankey_links_aggregated": {
        "data": get_sankey_links_aggregated(driver)
    },
    "zero_degree_nodes": {
        "data": get_zero_degree_node_stats(driver),
    },
}

save_kg_stats_json(
    kg_id=kg_id,
    kg_name=kg_name,
    stats_dict=stats_dict,
    out_path=out_path
)

driver.close()

Wrote statistics to monarchKgG_V4.4.42_stats.json


# Optimized Version!

In [5]:
from __future__ import annotations

import json
from typing import Any, Dict, List, Optional, Set, Tuple

from neo4j import GraphDatabase


# ---------------- CONFIG ----------------
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "password"  # TODO: change

# kg_id = "bioKG"
# kg_name = "bioKG"
# kg_id = "primeKG"
# kg_name = "Precision Medicine Knowledge Graph"
# kg_id = "PharMeBINet"
# kg_name = "heterogeneous pharmacological medical biochemical network"
# kg_id = "CKG (incomplete)"
# kg_name = "Clinical Knowledge Graph (incomplete)"
# kg_id = "OREGANO_V2"
# kg_name = "OREGANO"
# kg_id = "PetaGraph_V514"
# kg_name = "PetaGraph"
# kg_id = "BioDWH2_v4"
# kg_name = "Bio Data Warehouse 2"
# kg_id = "Hetionet"
# kg_name = "Hetionet"
# kg_id = "Bioteque"
# kg_name = "Bioteque"
# kg_id = "DRKG"
# kg_name = "Drug Repurposing Knowledge Graph"
# kg_id = "RTX-KG2"
# kg_name = "RTX-KG2"
kg_id = "Reactome"
kg_name = "Reactome_Oct_2025"
out_path = f"{kg_id}_stats.json"

# Optional: Connected Components (Neo4j GDS)
INCLUDE_GDS_WCC = True
GDS_GRAPH_NAME = "kg_cc"
TOP_COMPONENTS = 25

# Schema/property richness
INCLUDE_SCHEMA_PROPERTIES = True
TOP_PROPERTY_KEYS = 20
TOP_PROPERTY_KEYS_PER_LABEL = 10
# If you're on neo4j-driver v5+, fetch_size can help stream large results:
DRIVER_KWARGS = dict(fetch_size=2000)  # remove if your driver errors on it


# ---------------- QUERIES (NON-SANKEY) ----------------
QUERIES = {
    # Nodes by complete label-combination (multi-label combos)
    "node_count_per_type": """
        MATCH (n)
        RETURN labels(n) AS labels, count(*) AS count
        ORDER BY count DESC
    """,
    # Relationship counts by type (single scan)
    "relationship_count_per_type": """
        MATCH ()-[r]->()
        RETURN type(r) AS type, count(*) AS count
        ORDER BY count DESC
    """,
    # Degree distributions (overall)
    "out_degree_distribution": """
        MATCH (n)
        WITH count { (n)-->() } AS degree
        RETURN degree, count(*) AS count
        ORDER BY degree
    """,
    "in_degree_distribution": """
        MATCH (n)
        WITH count { ()-->(n) } AS degree
        RETURN degree, count(*) AS count
        ORDER BY degree
    """,
    "total_degree_distribution": """
        MATCH (n)
        WITH size([(n)-->() | 1]) + size([()-->(n) | 1]) AS degree
        RETURN degree, count(*) AS count
        ORDER BY degree
    """,
    # Degree distributions by label (no per-label loop)
    "out_degree_by_label": """
        MATCH (n)
        WITH n, count { (n)-->() } AS degree
        UNWIND labels(n) AS label
        RETURN label, degree, count(*) AS count
        ORDER BY label, degree
    """,
    "in_degree_by_label": """
        MATCH (n)
        WITH n, count { ()-->(n) } AS degree
        UNWIND labels(n) AS label
        RETURN label, degree, count(*) AS count
        ORDER BY label, degree
    """,
    "total_degree_by_label": """
        MATCH (n)
        WITH n, (size([(n)-->() | 1]) + size([()-->(n) | 1])) AS degree
        UNWIND labels(n) AS label
        RETURN label, degree, count(*) AS count
        ORDER BY label, degree
    """,
    # Degree distributions by label-combo (can be large, but no N+1)
    "out_degree_by_label_combo": """
        MATCH (n)
        WITH labels(n) AS labels, count { (n)-->() } AS degree
        RETURN labels, degree, count(*) AS count
        ORDER BY labels, degree
    """,
    "in_degree_by_label_combo": """
        MATCH (n)
        WITH labels(n) AS labels, count { ()-->(n) } AS degree
        RETURN labels, degree, count(*) AS count
        ORDER BY labels, degree
    """,
    "total_degree_by_label_combo": """
        MATCH (n)
        WITH labels(n) AS labels,
             (size([(n)-->() | 1]) + size([()-->(n) | 1])) AS degree
        RETURN labels, degree, count(*) AS count
        ORDER BY labels, degree
    """,
    # Zero-degree stats (single query)
    "zero_degree_nodes": """
        MATCH (n)
        WITH count(*) AS total,
             sum(CASE WHEN NOT (n)--() THEN 1 ELSE 0 END) AS zero
        RETURN
          zero AS zero_degree_nodes,
          total AS total_nodes,
          CASE WHEN total = 0 THEN 0.0 ELSE (toFloat(zero) / total) * 100 END AS percent
    """,
}


# ---------------- BASIC HELPERS ----------------
def run_list(session, query: str, params: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
    return [dict(r) for r in session.run(query, params or {})]


def reshape_node_counts_combo(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # [{"type":[...], "count":N}, ...]
    return [{"type": r["labels"], "count": r["count"]} for r in rows]


def reshape_rel_counts(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # [{"type":"REL", "count":N}, ...]
    return [{"type": r["type"], "count": r["count"]} for r in rows]


def reshape_degree_dist(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # [{"degree":d, "count":N}, ...]
    return [{"degree": r["degree"], "count": r["count"]} for r in rows]


def reshape_degree_by_label(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # rows: {label, degree, count} -> [{"label":..., "degrees":[{degree,count},...]}, ...]
    by_label: Dict[str, List[Dict[str, Any]]] = {}
    for r in rows:
        by_label.setdefault(r["label"], []).append({"degree": r["degree"], "count": r["count"]})
    return [{"label": label, "degrees": degs} for label, degs in by_label.items()]


def reshape_degree_by_combo(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # rows: {labels, degree, count} -> [{"labels":[...], "degrees":[{degree,count},...]}, ...]
    by_combo: Dict[Tuple[str, ...], List[Dict[str, Any]]] = {}
    for r in rows:
        key = tuple(r["labels"])
        by_combo.setdefault(key, []).append({"degree": r["degree"], "count": r["count"]})
    return [{"labels": list(labels), "degrees": degs} for labels, degs in by_combo.items()]


# ---------------- EXACT FRONTEND-MATCH AGGREGATION (buildAutoIdMap) ----------------
def simplicity_score(lab: str) -> int:
    # Matches frontend: slash*100 + pipe*100 + underscore*10 + length
    slash = lab.count("/")
    pipe = lab.count("|")
    underscore = lab.count("_")
    return slash * 100 + pipe * 100 + underscore * 10 + len(lab)


def build_auto_agg_maps(session) -> Tuple[Set[str], Dict[str, int], Dict[Tuple[str, str], int]]:
    """
    Build what frontend buildAutoIdMap() needs, without node-id sets:
    - singlePrimaries: labels that appear on some nodes with size(labels(n)) == 1
      (implemented via min(size(labels(n))) == 1)
    - label_count[label]
    - co[(a,b)] = count(nodes that have both labels a and b)
    """

    label_count: Dict[str, int] = {}
    single_primaries: Set[str] = set()

    rows = session.run("""
        MATCH (n)
        UNWIND labels(n) AS label
        RETURN label, count(*) AS cnt, min(size(labels(n))) AS minSize
    """)
    for r in rows:
        lab = r["label"]
        cnt = int(r["cnt"])
        min_size = int(r["minSize"]) if r["minSize"] is not None else 999999
        label_count[lab] = cnt
        if min_size == 1:
            single_primaries.add(lab)

    co: Dict[Tuple[str, str], int] = {}
    co_rows = session.run("""
        MATCH (n)
        WITH labels(n) AS labs
        UNWIND labs AS a
        UNWIND labs AS b
        RETURN a AS labelA, b AS labelB, count(*) AS co
    """)
    for r in co_rows:
        co[(r["labelA"], r["labelB"])] = int(r["co"])

    return single_primaries, label_count, co


def implies(label_other: str, label_cand: str,
            label_count: Dict[str, int],
            co: Dict[Tuple[str, str], int]) -> bool:
    """
    otherSet ⊆ candSet  <=>  count(other AND cand) == count(other)
    """
    other_cnt = label_count.get(label_other, 0)
    if other_cnt == 0:
        return False
    return co.get((label_other, label_cand), 0) == other_cnt


def pick_primary_exact(labels: List[str],
                       single_primaries: Set[str],
                       label_count: Dict[str, int],
                       co: Dict[Tuple[str, str], int]) -> str:
    """
    Mirrors frontend buildAutoIdMap() for a label list:
    - coversAll candidates: cand where every other label implies cand
    - else candidates = all labels
    - choose by: isSingle desc, size desc, simplicity asc
    """
    labs: List[str] = list(dict.fromkeys(labels or []))  # unique, stable order
    if len(labs) <= 1:
        return labs[0] if labs else "Unknown"

    covers_all: List[str] = []
    for cand in labs:
        ok = True
        for other in labs:
            if other == cand:
                continue
            if not implies(other, cand, label_count, co):
                ok = False
                break
        if ok:
            covers_all.append(cand)

    candidates = covers_all if covers_all else labs

    best = candidates[0]
    best_is_single = 1 if best in single_primaries else 0
    best_size = label_count.get(best, 0)
    best_simp = simplicity_score(best)

    for cand in candidates:
        is_single = 1 if cand in single_primaries else 0
        size = label_count.get(cand, 0)
        simp = simplicity_score(cand)

        if (
            is_single > best_is_single or
            (is_single == best_is_single and size > best_size) or
            (is_single == best_is_single and size == best_size and simp < best_simp)
        ):
            best = cand
            best_is_single = is_single
            best_size = size
            best_simp = simp

    return best


# ---------------- AGGREGATED NODE COUNTS (PRIMARY) ----------------
def get_node_counts_primary_from_combo_counts(
    combo_rows: List[Dict[str, Any]],
    single_primaries: Set[str],
    label_count: Dict[str, int],
    co: Dict[Tuple[str, str], int],
) -> List[Dict[str, Any]]:
    """
    Uses already-grouped combo counts (from node_count_per_type query) and maps each combo -> primary.
    Output:
      [{"type": ["PrimaryLabel"], "count": N}, ...]
    """
    agg: Dict[str, int] = {}
    for r in combo_rows:
        labels = r.get("labels") or []
        cnt = int(r.get("count") or 0)
        primary = pick_primary_exact(labels, single_primaries, label_count, co)
        agg[primary] = agg.get(primary, 0) + cnt

    return [{"type": [k], "count": v} for k, v in sorted(agg.items(), key=lambda kv: kv[1], reverse=True)]


# ---------------- SANKEY (AGGREGATED ONLY) ----------------
def get_sankey_links_aggregated_from_maps(
    session,
    single_primaries: Set[str],
    label_count: Dict[str, int],
    co: Dict[Tuple[str, str], int],
) -> List[Dict[str, Any]]:
    """
    Returns aggregated sankey links:
      [{source: string, relationship: string, target: string, value: int}, ...]
    Aggregation matches frontend auto mode mapping.
    """
    edge_rows = session.run("""
        MATCH (a)-[r]->(b)
        WITH labels(a) AS src_labels,
             type(r) AS relationship,
             labels(b) AS tgt_labels,
             count(*) AS value
        RETURN src_labels, relationship, tgt_labels, value
    """)

    combined: Dict[Tuple[str, str, str], int] = {}
    for r in edge_rows:
        src = pick_primary_exact(r["src_labels"], single_primaries, label_count, co)
        tgt = pick_primary_exact(r["tgt_labels"], single_primaries, label_count, co)
        rel = r["relationship"]
        val = int(r["value"])
        key = (src, rel, tgt)
        combined[key] = combined.get(key, 0) + val

    return [
        {"source": s, "relationship": rel, "target": t, "value": v}
        for (s, rel, t), v in sorted(combined.items(), key=lambda kv: kv[1], reverse=True)
    ]


# ---------------- GDS WCC (CONNECTED COMPONENTS) ----------------
def gds_connected_components_enhanced(session, graph_name: str, top_k: int = 25) -> Dict[str, Any]:
    gds_drop_if_exists(session, graph_name)

    # Projection (same as your working browser test)
    proj = session.run(
        "CALL gds.graph.project($name, '*', '*') "
        "YIELD graphName, nodeCount, relationshipCount "
        "RETURN graphName, nodeCount, relationshipCount",
        {"name": graph_name},
    ).single()
    if not proj:
        raise RuntimeError("GDS projection failed")

    node_count_total = int(proj["nodeCount"])
    rel_count_total = int(proj["relationshipCount"])

    # Compute component sizes once (still can be many rows, but 531k is usually OK offline)
    # We only RETURN aggregated info + top_k, not every component row to Python.
    cc_sizes = session.run(
        """
        CALL gds.wcc.stream($name)
        YIELD componentId
        RETURN componentId, count(*) AS size
        """,
        {"name": graph_name},
    )

    # Collect sizes in Python (componentCount can be large; still manageable offline)
    sizes: List[Tuple[int, int]] = []
    for r in cc_sizes:
        sizes.append((int(r["componentId"]), int(r["size"])))

    component_count = len(sizes)
    sizes_sorted = sorted(sizes, key=lambda x: x[1], reverse=True)

    top_components = [
        {"componentId": cid, "size": sz}
        for cid, sz in sizes_sorted[:top_k]
    ]

    giant_id, giant_size = sizes_sorted[0] if sizes_sorted else (None, 0)
    giant_pct = (giant_size / node_count_total * 100.0) if node_count_total else 0.0
    outside_nodes = max(0, node_count_total - giant_size)
    outside_pct = max(0.0, 100.0 - giant_pct)

    singleton_components = sum(1 for _, sz in sizes if sz == 1)
    singleton_nodes = singleton_components

    small_components_le_5 = sum(1 for _, sz in sizes if sz <= 5)
    small_nodes_le_5 = sum(sz for _, sz in sizes if sz <= 5)
    small_pct_le_5 = (small_nodes_le_5 / node_count_total * 100.0) if node_count_total else 0.0

    # Optional: histogram buckets
    buckets = {"1": 0, "2": 0, "3-5": 0, "6-10": 0, "11-100": 0, ">100": 0}
    for _, sz in sizes:
        if sz == 1: buckets["1"] += 1
        elif sz == 2: buckets["2"] += 1
        elif 3 <= sz <= 5: buckets["3-5"] += 1
        elif 6 <= sz <= 10: buckets["6-10"] += 1
        elif 11 <= sz <= 100: buckets["11-100"] += 1
        else: buckets[">100"] += 1

    out: Dict[str, Any] = {
        "graph_name": proj["graphName"],
        "nodeCount": node_count_total,
        "relationshipCount": rel_count_total,
        "componentCount": component_count,
        "topComponents": top_components,

        # summary extras
        "giantComponentId": giant_id,
        "giantSize": giant_size,
        "giantPercent": giant_pct,
        "nodesOutsideGiant": outside_nodes,
        "outsidePercent": outside_pct,
        "singletonComponentCount": singleton_components,
        "singletonNodes": singleton_nodes,
        "smallComponentCount_le_5": small_components_le_5,
        "smallNodes_le_5": small_nodes_le_5,
        "smallPercent_le_5": small_pct_le_5,
        "sizeHistogram": buckets,
    }
    try:
        session.run("CALL gds.graph.drop($name, false) YIELD graphName", {"name": graph_name}).consume()
    except Exception:
        pass
    return out


def gds_drop_if_exists(session, name: str) -> None:
    # Newer GDS supports: gds.graph.drop(name, false)
    try:
        session.run("CALL gds.graph.drop($name, false) YIELD graphName", {"name": name}).consume()
        return
    except Exception:
        pass

    # Fallback: exists + conditional drop (no CASE CALL)
    try:
        rec = session.run(
            "CALL gds.graph.exists($name) YIELD exists RETURN exists",
            {"name": name},
        ).single()
        if rec and rec["exists"]:
            session.run("CALL gds.graph.drop($name) YIELD graphName", {"name": name}).consume()
    except Exception:
        # If exists procedure doesn't exist in your GDS, ignore
        return

def schema_properties_summary(session, top_keys: int = 20, top_per_label: int = 10) -> Dict[str, Any]:
    # Distinct property keys overall
    rec = session.run("CALL db.propertyKeys() YIELD propertyKey RETURN collect(propertyKey) AS keys").single()
    keys = rec["keys"] if rec else []
    key_count = len(keys)

    # Top property keys by frequency
    top_key_rows = run_list(
        session,
        """
        MATCH (n)
        UNWIND keys(n) AS key
        RETURN key AS key, count(*) AS count
        ORDER BY count DESC
        LIMIT $limit
        """,
        {"limit": int(top_keys)},
    )

    # Avg property keys per node + % nodes with any property
    rec2 = session.run(
        """
        MATCH (n)
        WITH count(n) AS total,
             avg(size(keys(n))) AS avgKeys,
             sum(CASE WHEN size(keys(n)) > 0 THEN 1 ELSE 0 END) AS withProps
        RETURN total, avgKeys, withProps
        """
    ).single()

    total_nodes = int(rec2["total"]) if rec2 else 0
    avg_keys = float(rec2["avgKeys"]) if rec2 and rec2["avgKeys"] is not None else 0.0
    with_props = int(rec2["withProps"]) if rec2 else 0
    with_props_pct = (with_props / total_nodes * 100.0) if total_nodes else 0.0

    # Top keys per label (limit)
    per_label_rows = run_list(
        session,
        """
        MATCH (n)
        UNWIND labels(n) AS label
        UNWIND keys(n) AS k
        RETURN label, k AS key, count(*) AS count
        ORDER BY label, count DESC
        """,
    )

    by_label: Dict[str, List[Dict[str, Any]]] = {}
    for r in per_label_rows:
        lab = r["label"]
        by_label.setdefault(lab, [])
        if len(by_label[lab]) < top_per_label:
            by_label[lab].append({"key": r["key"], "count": int(r["count"])})

    return {
        "propertyKeyCount": key_count,
        "propertyKeyTopN": [{"key": r["key"], "count": int(r["count"])} for r in top_key_rows],
        "avgPropertyKeysPerNode": avg_keys,
        "nodesWithAnyPropertiesPercent": with_props_pct,
        "propertyKeysByLabelTopN": [{"label": lab, "topKeys": lst} for lab, lst in by_label.items()],
    }


# ---------------- MAIN EXTRACTION ----------------
def extract_stats(driver) -> Dict[str, Any]:
    stats_dict: Dict[str, Any] = {}

    with driver.session() as session:
        # Build frontend-equivalent aggregation maps ONCE (used for sankey + node_count_primary)
        single_primaries, label_count, co = build_auto_agg_maps(session)

        # Node count by label combo (also reused for node_count_primary)
        combo_rows = run_list(session, QUERIES["node_count_per_type"])
        stats_dict["node_count_per_type"] = {"data": reshape_node_counts_combo(combo_rows)}

        # Aggregated node counts by primary label (multilabel-friendly)
        stats_dict["node_count_primary"] = {
            "data": get_node_counts_primary_from_combo_counts(combo_rows, single_primaries, label_count, co)
        }

        # Relationship counts
        rows = run_list(session, QUERIES["relationship_count_per_type"])
        stats_dict["relationship_count_per_type"] = {"data": reshape_rel_counts(rows)}

        # Overall degree distributions
        rows = run_list(session, QUERIES["out_degree_distribution"])
        stats_dict["out_degree_distribution"] = {"data": reshape_degree_dist(rows)}

        rows = run_list(session, QUERIES["in_degree_distribution"])
        stats_dict["in_degree_distribution"] = {"data": reshape_degree_dist(rows)}

        rows = run_list(session, QUERIES["total_degree_distribution"])
        stats_dict["total_degree_distribution"] = {"data": reshape_degree_dist(rows)}

        # Degree by label
        rows = run_list(session, QUERIES["out_degree_by_label"])
        stats_dict["out_degree_by_label"] = {"data": reshape_degree_by_label(rows)}

        rows = run_list(session, QUERIES["in_degree_by_label"])
        stats_dict["in_degree_by_label"] = {"data": reshape_degree_by_label(rows)}

        rows = run_list(session, QUERIES["total_degree_by_label"])
        stats_dict["total_degree_by_label"] = {"data": reshape_degree_by_label(rows)}

        # Degree by label combo
        rows = run_list(session, QUERIES["out_degree_by_label_combo"])
        stats_dict["out_degree_by_label_combo"] = {"data": reshape_degree_by_combo(rows)}

        rows = run_list(session, QUERIES["in_degree_by_label_combo"])
        stats_dict["in_degree_by_label_combo"] = {"data": reshape_degree_by_combo(rows)}

        rows = run_list(session, QUERIES["total_degree_by_label_combo"])
        stats_dict["total_degree_by_label_combo"] = {"data": reshape_degree_by_combo(rows)}

        # Zero-degree stats
        row = session.run(QUERIES["zero_degree_nodes"]).single()
        stats_dict["zero_degree_nodes"] = {"data": dict(row) if row else {}}

        # Sankey aggregated (only)
        stats_dict["sankey_links_aggregated"] = {
            "data": get_sankey_links_aggregated_from_maps(session, single_primaries, label_count, co)
        }

       # Connected components enhanced
        if INCLUDE_GDS_WCC:
            try:
                cc = gds_connected_components_enhanced(session, GDS_GRAPH_NAME, TOP_COMPONENTS)
                stats_dict["connected_components"] = {"data": cc}
        
            except Exception as e:
                stats_dict["connected_components"] = {"data": {"error": str(e)}}

        if INCLUDE_SCHEMA_PROPERTIES:
            try:
                sp = schema_properties_summary(session, TOP_PROPERTY_KEYS, TOP_PROPERTY_KEYS_PER_LABEL)
                stats_dict["schema_properties_summary"] = {"data": sp}
            except Exception as e:
                stats_dict["schema_properties_summary"] = {"data": {"error": str(e)}}

    return stats_dict


def save_stats_json(
    kg_id: str,
    kg_name: str,
    stats_dict: Dict[str, Any],
    out_path: str,
) -> None:
    stats_out = [{"stat_id": stat_id, "data": stat.get("data")} for stat_id, stat in stats_dict.items()]
    out = {"kg_id": kg_id, "kg_name": kg_name, "stats": stats_out}

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

    print(f"Wrote statistics to {out_path}")


def main() -> None:
    # Connect
    try:
        driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(NEO4J_USER, NEO4J_PASS),
            **DRIVER_KWARGS,
        )
    except TypeError:
        # Older driver versions may not accept fetch_size in constructor
        driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(NEO4J_USER, NEO4J_PASS),
        )

    try:
        stats_dict = extract_stats(driver)
        save_stats_json(kg_id=kg_id, kg_name=kg_name, stats_dict=stats_dict, out_path=out_path)
    finally:
        driver.close()


if __name__ == "__main__":
    main()

CypherSyntaxError: {code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input '(': expected ",", ".", "}" or an identifier (line 3, column 22 (offset: 40))
"        WITH count { (n)-->() } AS degree"
                      ^}

Older Versions 4.XX Neo4J

In [1]:
from __future__ import annotations

import json
from typing import Any, Dict, List, Optional, Set, Tuple

from neo4j import GraphDatabase


# ---------------- CONFIG ----------------
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "password"  # TODO: change

# kg_id = "bioKG"
# kg_name = "bioKG"
# kg_id = "primeKG"
# kg_name = "Precision Medicine Knowledge Graph"
# kg_id = "PharMeBINet"
# kg_name = "heterogeneous pharmacological medical biochemical network"
# kg_id = "CKG (incomplete)"
# kg_name = "Clinical Knowledge Graph (incomplete)"
kg_id = "CKG"
kg_name = "Clinical Knowledge Graph"
# kg_id = "OREGANO_V2"
# kg_name = "OREGANO"
# kg_id = "PetaGraph_V514"
# kg_name = "PetaGraph"
# kg_id = "BioDWH2_v4"
# kg_name = "Bio Data Warehouse 2"
# kg_id = "Hetionet"
# kg_name = "Hetionet"
# kg_id = "Bioteque"
# kg_name = "Bioteque"
# kg_id = "DRKG"
# kg_name = "Drug Repurposing Knowledge Graph"
# kg_id = "RTX-KG2"
# kg_name = "RTX-KG2"
# kg_id = "Reactome"
# kg_name = "Reactome_Oct_2025"
# kg_id = "Monarch-KG"
# kg_name = "Monarch-KG V4.4"
out_path = f"{kg_id}_stats.json"

# Optional: Connected Components (Neo4j GDS)
INCLUDE_GDS_WCC = True
GDS_GRAPH_NAME = "kg_cc"
TOP_COMPONENTS = 25

# Schema/property richness
INCLUDE_SCHEMA_PROPERTIES = True
TOP_PROPERTY_KEYS = 20
TOP_PROPERTY_KEYS_PER_LABEL = 10
# If you're on neo4j-driver v5+, fetch_size can help stream large results:
DRIVER_KWARGS = dict(fetch_size=2000)  # remove if your driver errors on it


# ---------------- QUERIES (NON-SANKEY) ----------------
QUERIES = {
    # Nodes by complete label-combination (multi-label combos)
    "node_count_per_type": """
        MATCH (n)
        RETURN labels(n) AS labels, count(*) AS count
        ORDER BY count DESC
    """,
    # Relationship counts by type (single scan)
    "relationship_count_per_type": """
        MATCH ()-[r]->()
        RETURN type(r) AS type, count(*) AS count
        ORDER BY count DESC
    """,
    # Degree distributions (overall)
    "out_degree_distribution": """
        MATCH (n)
        OPTIONAL MATCH (n)-->()
        WITH n, count(*) AS degree
        RETURN degree, count(*) AS count
        ORDER BY degree
    """,
    "in_degree_distribution": """
        MATCH (n)
        OPTIONAL MATCH ()-->(n)
        WITH n, count(*) AS degree
        RETURN degree, count(*) AS count
        ORDER BY degree
    """,
    "total_degree_distribution": """
        MATCH (n)
        OPTIONAL MATCH (n)--(m)
        WITH n, count(m) AS degree
        RETURN degree, count(*) AS count
        ORDER BY degree
    """,
    # Degree distributions by label (no per-label loop)
    "out_degree_by_label": """
        MATCH (n)
        OPTIONAL MATCH (n)-->()
        WITH n, count(*) AS degree
        UNWIND labels(n) AS label
        RETURN label, degree, count(*) AS count
        ORDER BY label, degree
    """,
    "in_degree_by_label": """
        MATCH (n)
        OPTIONAL MATCH ()-->(n)
        WITH n, count(*) AS degree
        UNWIND labels(n) AS label
        RETURN label, degree, count(*) AS count
        ORDER BY label, degree
    """,
    "total_degree_by_label": """
        MATCH (n)
        OPTIONAL MATCH (n)--(m)
        WITH n, count(m) AS degree
        UNWIND labels(n) AS label
        RETURN label, degree, count(*) AS count
        ORDER BY label, degree
    """,
    # Degree distributions by label-combo (can be large, but no N+1)
    "out_degree_by_label_combo": """
        MATCH (n)
        OPTIONAL MATCH (n)-->()
        WITH labels(n) AS labels, count(*) AS degree
        RETURN labels, degree, count(*) AS count
        ORDER BY labels, degree
    """,
    "in_degree_by_label_combo": """
        MATCH (n)
        OPTIONAL MATCH ()-->(n)
        WITH labels(n) AS labels, count(*) AS degree
        RETURN labels, degree, count(*) AS count
        ORDER BY labels, degree
    """,
    "total_degree_by_label_combo": """
        MATCH (n)
        OPTIONAL MATCH (n)--(m)
        WITH labels(n) AS labels, count(m) AS degree
        RETURN labels, degree, count(*) AS count
        ORDER BY labels, degree
    """,
    # Zero-degree stats (single query)
    "zero_degree_nodes": """
        MATCH (n)
        WITH count(*) AS total,
             sum(CASE WHEN NOT (n)--() THEN 1 ELSE 0 END) AS zero
        RETURN
          zero AS zero_degree_nodes,
          total AS total_nodes,
          CASE WHEN total = 0 THEN 0.0 ELSE (toFloat(zero) / total) * 100 END AS percent
    """,
}


# ---------------- BASIC HELPERS ----------------
def run_list(session, query: str, params: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
    return [dict(r) for r in session.run(query, params or {})]


def reshape_node_counts_combo(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # [{"type":[...], "count":N}, ...]
    return [{"type": r["labels"], "count": r["count"]} for r in rows]


def reshape_rel_counts(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # [{"type":"REL", "count":N}, ...]
    return [{"type": r["type"], "count": r["count"]} for r in rows]


def reshape_degree_dist(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # [{"degree":d, "count":N}, ...]
    return [{"degree": r["degree"], "count": r["count"]} for r in rows]


def reshape_degree_by_label(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # rows: {label, degree, count} -> [{"label":..., "degrees":[{degree,count},...]}, ...]
    by_label: Dict[str, List[Dict[str, Any]]] = {}
    for r in rows:
        by_label.setdefault(r["label"], []).append({"degree": r["degree"], "count": r["count"]})
    return [{"label": label, "degrees": degs} for label, degs in by_label.items()]


def reshape_degree_by_combo(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # rows: {labels, degree, count} -> [{"labels":[...], "degrees":[{degree,count},...]}, ...]
    by_combo: Dict[Tuple[str, ...], List[Dict[str, Any]]] = {}
    for r in rows:
        key = tuple(r["labels"])
        by_combo.setdefault(key, []).append({"degree": r["degree"], "count": r["count"]})
    return [{"labels": list(labels), "degrees": degs} for labels, degs in by_combo.items()]


# ---------------- EXACT FRONTEND-MATCH AGGREGATION (buildAutoIdMap) ----------------
def simplicity_score(lab: str) -> int:
    # Matches frontend: slash*100 + pipe*100 + underscore*10 + length
    slash = lab.count("/")
    pipe = lab.count("|")
    underscore = lab.count("_")
    return slash * 100 + pipe * 100 + underscore * 10 + len(lab)


def build_auto_agg_maps(session) -> Tuple[Set[str], Dict[str, int], Dict[Tuple[str, str], int]]:
    """
    Build what frontend buildAutoIdMap() needs, without node-id sets:
    - singlePrimaries: labels that appear on some nodes with size(labels(n)) == 1
      (implemented via min(size(labels(n))) == 1)
    - label_count[label]
    - co[(a,b)] = count(nodes that have both labels a and b)
    """

    label_count: Dict[str, int] = {}
    single_primaries: Set[str] = set()

    rows = session.run("""
        MATCH (n)
        UNWIND labels(n) AS label
        RETURN label, count(*) AS cnt, min(size(labels(n))) AS minSize
    """)
    for r in rows:
        lab = r["label"]
        cnt = int(r["cnt"])
        min_size = int(r["minSize"]) if r["minSize"] is not None else 999999
        label_count[lab] = cnt
        if min_size == 1:
            single_primaries.add(lab)

    co: Dict[Tuple[str, str], int] = {}
    co_rows = session.run("""
        MATCH (n)
        WITH labels(n) AS labs
        UNWIND labs AS a
        UNWIND labs AS b
        RETURN a AS labelA, b AS labelB, count(*) AS co
    """)
    for r in co_rows:
        co[(r["labelA"], r["labelB"])] = int(r["co"])

    return single_primaries, label_count, co


def implies(label_other: str, label_cand: str,
            label_count: Dict[str, int],
            co: Dict[Tuple[str, str], int]) -> bool:
    """
    otherSet ⊆ candSet  <=>  count(other AND cand) == count(other)
    """
    other_cnt = label_count.get(label_other, 0)
    if other_cnt == 0:
        return False
    return co.get((label_other, label_cand), 0) == other_cnt


def pick_primary_exact(labels: List[str],
                       single_primaries: Set[str],
                       label_count: Dict[str, int],
                       co: Dict[Tuple[str, str], int]) -> str:
    """
    Mirrors frontend buildAutoIdMap() for a label list:
    - coversAll candidates: cand where every other label implies cand
    - else candidates = all labels
    - choose by: isSingle desc, size desc, simplicity asc
    """
    labs: List[str] = list(dict.fromkeys(labels or []))  # unique, stable order
    if len(labs) <= 1:
        return labs[0] if labs else "Unknown"

    covers_all: List[str] = []
    for cand in labs:
        ok = True
        for other in labs:
            if other == cand:
                continue
            if not implies(other, cand, label_count, co):
                ok = False
                break
        if ok:
            covers_all.append(cand)

    candidates = covers_all if covers_all else labs

    best = candidates[0]
    best_is_single = 1 if best in single_primaries else 0
    best_size = label_count.get(best, 0)
    best_simp = simplicity_score(best)

    for cand in candidates:
        is_single = 1 if cand in single_primaries else 0
        size = label_count.get(cand, 0)
        simp = simplicity_score(cand)

        if (
            is_single > best_is_single or
            (is_single == best_is_single and size > best_size) or
            (is_single == best_is_single and size == best_size and simp < best_simp)
        ):
            best = cand
            best_is_single = is_single
            best_size = size
            best_simp = simp

    return best


# ---------------- AGGREGATED NODE COUNTS (PRIMARY) ----------------
def get_node_counts_primary_from_combo_counts(
    combo_rows: List[Dict[str, Any]],
    single_primaries: Set[str],
    label_count: Dict[str, int],
    co: Dict[Tuple[str, str], int],
) -> List[Dict[str, Any]]:
    """
    Uses already-grouped combo counts (from node_count_per_type query) and maps each combo -> primary.
    Output:
      [{"type": ["PrimaryLabel"], "count": N}, ...]
    """
    agg: Dict[str, int] = {}
    for r in combo_rows:
        labels = r.get("labels") or []
        cnt = int(r.get("count") or 0)
        primary = pick_primary_exact(labels, single_primaries, label_count, co)
        agg[primary] = agg.get(primary, 0) + cnt

    return [{"type": [k], "count": v} for k, v in sorted(agg.items(), key=lambda kv: kv[1], reverse=True)]


# ---------------- SANKEY (AGGREGATED ONLY) ----------------
def get_sankey_links_aggregated_from_maps(
    session,
    single_primaries: Set[str],
    label_count: Dict[str, int],
    co: Dict[Tuple[str, str], int],
) -> List[Dict[str, Any]]:
    """
    Returns aggregated sankey links:
      [{source: string, relationship: string, target: string, value: int}, ...]
    Aggregation matches frontend auto mode mapping.
    """
    edge_rows = session.run("""
        MATCH (a)-[r]->(b)
        WITH labels(a) AS src_labels,
             type(r) AS relationship,
             labels(b) AS tgt_labels,
             count(*) AS value
        RETURN src_labels, relationship, tgt_labels, value
    """)

    combined: Dict[Tuple[str, str, str], int] = {}
    for r in edge_rows:
        src = pick_primary_exact(r["src_labels"], single_primaries, label_count, co)
        tgt = pick_primary_exact(r["tgt_labels"], single_primaries, label_count, co)
        rel = r["relationship"]
        val = int(r["value"])
        key = (src, rel, tgt)
        combined[key] = combined.get(key, 0) + val

    return [
        {"source": s, "relationship": rel, "target": t, "value": v}
        for (s, rel, t), v in sorted(combined.items(), key=lambda kv: kv[1], reverse=True)
    ]


# ---------------- GDS WCC (CONNECTED COMPONENTS) ----------------
def gds_connected_components_enhanced(session, graph_name: str, top_k: int = 25) -> Dict[str, Any]:
    gds_drop_if_exists(session, graph_name)

    # Projection (same as your working browser test)
    proj = session.run(
        "CALL gds.graph.project($name, '*', '*') "
        "YIELD graphName, nodeCount, relationshipCount "
        "RETURN graphName, nodeCount, relationshipCount",
        {"name": graph_name},
    ).single()
    if not proj:
        raise RuntimeError("GDS projection failed")

    node_count_total = int(proj["nodeCount"])
    rel_count_total = int(proj["relationshipCount"])

    # Compute component sizes once (still can be many rows, but 531k is usually OK offline)
    # We only RETURN aggregated info + top_k, not every component row to Python.
    cc_sizes = session.run(
        """
        CALL gds.wcc.stream($name)
        YIELD componentId
        RETURN componentId, count(*) AS size
        """,
        {"name": graph_name},
    )

    # Collect sizes in Python (componentCount can be large; still manageable offline)
    sizes: List[Tuple[int, int]] = []
    for r in cc_sizes:
        sizes.append((int(r["componentId"]), int(r["size"])))

    component_count = len(sizes)
    sizes_sorted = sorted(sizes, key=lambda x: x[1], reverse=True)

    top_components = [
        {"componentId": cid, "size": sz}
        for cid, sz in sizes_sorted[:top_k]
    ]

    giant_id, giant_size = sizes_sorted[0] if sizes_sorted else (None, 0)
    giant_pct = (giant_size / node_count_total * 100.0) if node_count_total else 0.0
    outside_nodes = max(0, node_count_total - giant_size)
    outside_pct = max(0.0, 100.0 - giant_pct)

    singleton_components = sum(1 for _, sz in sizes if sz == 1)
    singleton_nodes = singleton_components

    small_components_le_5 = sum(1 for _, sz in sizes if sz <= 5)
    small_nodes_le_5 = sum(sz for _, sz in sizes if sz <= 5)
    small_pct_le_5 = (small_nodes_le_5 / node_count_total * 100.0) if node_count_total else 0.0

    # Optional: histogram buckets
    buckets = {"1": 0, "2": 0, "3-5": 0, "6-10": 0, "11-100": 0, ">100": 0}
    for _, sz in sizes:
        if sz == 1: buckets["1"] += 1
        elif sz == 2: buckets["2"] += 1
        elif 3 <= sz <= 5: buckets["3-5"] += 1
        elif 6 <= sz <= 10: buckets["6-10"] += 1
        elif 11 <= sz <= 100: buckets["11-100"] += 1
        else: buckets[">100"] += 1

    out: Dict[str, Any] = {
        "graph_name": proj["graphName"],
        "nodeCount": node_count_total,
        "relationshipCount": rel_count_total,
        "componentCount": component_count,
        "topComponents": top_components,

        # summary extras
        "giantComponentId": giant_id,
        "giantSize": giant_size,
        "giantPercent": giant_pct,
        "nodesOutsideGiant": outside_nodes,
        "outsidePercent": outside_pct,
        "singletonComponentCount": singleton_components,
        "singletonNodes": singleton_nodes,
        "smallComponentCount_le_5": small_components_le_5,
        "smallNodes_le_5": small_nodes_le_5,
        "smallPercent_le_5": small_pct_le_5,
        "sizeHistogram": buckets,
    }
    try:
        session.run("CALL gds.graph.drop($name, false) YIELD graphName", {"name": graph_name}).consume()
    except Exception:
        pass
    return out


def gds_drop_if_exists(session, name: str) -> None:
    # Newer GDS supports: gds.graph.drop(name, false)
    try:
        session.run("CALL gds.graph.drop($name, false) YIELD graphName", {"name": name}).consume()
        return
    except Exception:
        pass

    # Fallback: exists + conditional drop (no CASE CALL)
    try:
        rec = session.run(
            "CALL gds.graph.exists($name) YIELD exists RETURN exists",
            {"name": name},
        ).single()
        if rec and rec["exists"]:
            session.run("CALL gds.graph.drop($name) YIELD graphName", {"name": name}).consume()
    except Exception:
        # If exists procedure doesn't exist in your GDS, ignore
        return

def schema_properties_summary(session, top_keys: int = 20, top_per_label: int = 10) -> Dict[str, Any]:
    # Distinct property keys overall
    rec = session.run("CALL db.propertyKeys() YIELD propertyKey RETURN collect(propertyKey) AS keys").single()
    keys = rec["keys"] if rec else []
    key_count = len(keys)

    # Top property keys by frequency
    top_key_rows = run_list(
        session,
        """
        MATCH (n)
        UNWIND keys(n) AS key
        RETURN key AS key, count(*) AS count
        ORDER BY count DESC
        LIMIT $limit
        """,
        {"limit": int(top_keys)},
    )

    # Avg property keys per node + % nodes with any property
    rec2 = session.run(
        """
        MATCH (n)
        WITH count(n) AS total,
             avg(size(keys(n))) AS avgKeys,
             sum(CASE WHEN size(keys(n)) > 0 THEN 1 ELSE 0 END) AS withProps
        RETURN total, avgKeys, withProps
        """
    ).single()

    total_nodes = int(rec2["total"]) if rec2 else 0
    avg_keys = float(rec2["avgKeys"]) if rec2 and rec2["avgKeys"] is not None else 0.0
    with_props = int(rec2["withProps"]) if rec2 else 0
    with_props_pct = (with_props / total_nodes * 100.0) if total_nodes else 0.0

    # Top keys per label (limit)
    per_label_rows = run_list(
        session,
        """
        MATCH (n)
        UNWIND labels(n) AS label
        UNWIND keys(n) AS k
        RETURN label, k AS key, count(*) AS count
        ORDER BY label, count DESC
        """,
    )

    by_label: Dict[str, List[Dict[str, Any]]] = {}
    for r in per_label_rows:
        lab = r["label"]
        by_label.setdefault(lab, [])
        if len(by_label[lab]) < top_per_label:
            by_label[lab].append({"key": r["key"], "count": int(r["count"])})

    return {
        "propertyKeyCount": key_count,
        "propertyKeyTopN": [{"key": r["key"], "count": int(r["count"])} for r in top_key_rows],
        "avgPropertyKeysPerNode": avg_keys,
        "nodesWithAnyPropertiesPercent": with_props_pct,
        "propertyKeysByLabelTopN": [{"label": lab, "topKeys": lst} for lab, lst in by_label.items()],
    }


# ---------------- MAIN EXTRACTION ----------------
def extract_stats(driver) -> Dict[str, Any]:
    stats_dict: Dict[str, Any] = {}

    with driver.session() as session:
        # Build frontend-equivalent aggregation maps ONCE (used for sankey + node_count_primary)
        single_primaries, label_count, co = build_auto_agg_maps(session)

        # Node count by label combo (also reused for node_count_primary)
        combo_rows = run_list(session, QUERIES["node_count_per_type"])
        stats_dict["node_count_per_type"] = {"data": reshape_node_counts_combo(combo_rows)}

        # Aggregated node counts by primary label (multilabel-friendly)
        stats_dict["node_count_primary"] = {
            "data": get_node_counts_primary_from_combo_counts(combo_rows, single_primaries, label_count, co)
        }

        # Relationship counts
        rows = run_list(session, QUERIES["relationship_count_per_type"])
        stats_dict["relationship_count_per_type"] = {"data": reshape_rel_counts(rows)}

        # Overall degree distributions
        rows = run_list(session, QUERIES["out_degree_distribution"])
        stats_dict["out_degree_distribution"] = {"data": reshape_degree_dist(rows)}

        rows = run_list(session, QUERIES["in_degree_distribution"])
        stats_dict["in_degree_distribution"] = {"data": reshape_degree_dist(rows)}

        rows = run_list(session, QUERIES["total_degree_distribution"])
        stats_dict["total_degree_distribution"] = {"data": reshape_degree_dist(rows)}

        # Degree by label
        rows = run_list(session, QUERIES["out_degree_by_label"])
        stats_dict["out_degree_by_label"] = {"data": reshape_degree_by_label(rows)}

        rows = run_list(session, QUERIES["in_degree_by_label"])
        stats_dict["in_degree_by_label"] = {"data": reshape_degree_by_label(rows)}

        rows = run_list(session, QUERIES["total_degree_by_label"])
        stats_dict["total_degree_by_label"] = {"data": reshape_degree_by_label(rows)}

        # Degree by label combo
        rows = run_list(session, QUERIES["out_degree_by_label_combo"])
        stats_dict["out_degree_by_label_combo"] = {"data": reshape_degree_by_combo(rows)}

        rows = run_list(session, QUERIES["in_degree_by_label_combo"])
        stats_dict["in_degree_by_label_combo"] = {"data": reshape_degree_by_combo(rows)}

        rows = run_list(session, QUERIES["total_degree_by_label_combo"])
        stats_dict["total_degree_by_label_combo"] = {"data": reshape_degree_by_combo(rows)}

        # Zero-degree stats
        row = session.run(QUERIES["zero_degree_nodes"]).single()
        stats_dict["zero_degree_nodes"] = {"data": dict(row) if row else {}}

        # Sankey aggregated (only)
        stats_dict["sankey_links_aggregated"] = {
            "data": get_sankey_links_aggregated_from_maps(session, single_primaries, label_count, co)
        }

       # Connected components enhanced
        if INCLUDE_GDS_WCC:
            try:
                cc = gds_connected_components_enhanced(session, GDS_GRAPH_NAME, TOP_COMPONENTS)
                stats_dict["connected_components"] = {"data": cc}
        
            except Exception as e:
                stats_dict["connected_components"] = {"data": {"error": str(e)}}

        if INCLUDE_SCHEMA_PROPERTIES:
            try:
                sp = schema_properties_summary(session, TOP_PROPERTY_KEYS, TOP_PROPERTY_KEYS_PER_LABEL)
                stats_dict["schema_properties_summary"] = {"data": sp}
            except Exception as e:
                stats_dict["schema_properties_summary"] = {"data": {"error": str(e)}}

    return stats_dict


def save_stats_json(
    kg_id: str,
    kg_name: str,
    stats_dict: Dict[str, Any],
    out_path: str,
) -> None:
    stats_out = [{"stat_id": stat_id, "data": stat.get("data")} for stat_id, stat in stats_dict.items()]
    out = {"kg_id": kg_id, "kg_name": kg_name, "stats": stats_out}

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

    print(f"Wrote statistics to {out_path}")


def main() -> None:
    # Connect
    try:
        driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(NEO4J_USER, NEO4J_PASS),
            **DRIVER_KWARGS,
        )
    except TypeError:
        # Older driver versions may not accept fetch_size in constructor
        driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(NEO4J_USER, NEO4J_PASS),
        )

    try:
        stats_dict = extract_stats(driver)
        save_stats_json(kg_id=kg_id, kg_name=kg_name, stats_dict=stats_dict, out_path=out_path)
    finally:
        driver.close()


if __name__ == "__main__":
    main()

Wrote statistics to Reactome_stats.json
